In [ ]:
from numpy._core.defchararray import count
import pandas as pd
import numpy as np

df = pd.read_csv('netflix_titles.csv')

df["title"] = df["title"].str.replace("\xa0", " ", regex = False).str.strip()

compare_columns = [c for c in df.columns if c not in ("show_id",)]

duplicate_title = df.duplicated(subset = compare_columns, keep = "first")

#print(f"Exact duplicated removed: {duplicate_title.sum()}")

df = df[~duplicate_title].copy()

filter_min = df["rating"].str.contains("min", na = False)

#print(f"Rating/Duration shift: {filter_min.sum()} rows")

df.loc[filter_min, "duration"] = df.loc[filter_min, "rating"]

df.loc[filter_min, "rating"] = pd.NA

df["date_added"] = df["date_added"].str.strip()

df["date_added"] = pd.to_datetime(df["date_added"], errors= "coerce")

df["day_added"] = df["date_added"].dt.day.astype("Int64")

df["month_added"] = df["date_added"].dt.month.astype("Int64")

df["year_added"] = df["date_added"].dt.year.astype("Int64")

#df

titles = df[["show_id", "type", "title", "director", "cast", "country", "date_added",
             "day_added", "month_added", "year_added", "release_year", "rating", "duration", "listed_in", "description",]].copy()

#titles

assert titles["show_id"].is_unique, "Must be one row per show_id"

#print(f"Base Titles: {len(titles)} rows should equal to unique show_id")

title_genre = titles[["show_id", "listed_in"]].copy()

#title_genre

title_genre["listed_in"] = title_genre["listed_in"].str.split(",")

title_genre = title_genre.explode("listed_in")

title_genre["listed_in"] = title_genre["listed_in"].str.strip()

title_genre = title_genre.rename(columns = {"listed_in": "genre"})

title_genre = title_genre[title_genre["genre"].notna() & (title_genre["genre"] != "")]

before = len(title_genre)

title_genre = title_genre[~title_genre["genre"].isin(["Movies", "TV Shows"])]

#print(f"Generic genre tags removed: {before - len(title_genre)}")

#print(f"Table still has: {titles['show_id'].nunique()} shows"
      #f"-- {titles['show_id'].nunique() - title_genre['show_id'].nunique()} "
      #f"removed generic genre")

title_country = titles[["show_id", "country"]].copy()

title_country["country"] = title_country["country"].str.split(",")

title_country = title_country.explode("country")

title_country["country"] = title_country["country"].str.strip()

title_country = title_country[title_country["country"].notna() & (title_country["country"] != "")]

titles["primary_country"] = (titles["country"].str.split(",").str[0].str.strip())

print("Row count sanity check:")

print(f"  titles:        {len(titles)} (1 row per show)")

print(f"  title_genre:   {len(title_genre)} (should be << titles x avg genres, "
      f"NOT titles x avg genres x avg countries)")

print(f"  title_country: {len(title_country)}")

max_mult = (title_genre.groupby('show_id').size().max())

print(f"  max genre rows for a single title: {max_mult} (was 24 with the cartesian join)")

#titles.to_csv("netflix_titles_base.csv", index=False)

#title_genre.to_csv("netflix_title_genre.csv", index=False)

#title_country.to_csv("netflix_title_country.csv", index=False)

,show_id,country
0,s1,United States
1,s2,South Africa
4,s5,India
7,s8,United States
7,s8,Ghana
...,...,...
8801,s8802,Jordan
8802,s8803,United States
8804,s8805,United States
8805,s8806,United States
